In [2]:
import pandas as pd
import numpy as np

filepath = "NYPD_Complaint_Data_Current_(Year_To_Date)_20260317.csv"

# Load the dataset with CMPLNT_NUM as string to preserve leading zeros
df = pd.read_csv(filepath, dtype={'CMPLNT_NUM': str})

# Replace common placeholders for missing values with NaN
df = df.replace(["(null)", "", " "], np.nan)



In [4]:
# Check date columns 
date_cols = ["CMPLNT_FR_DT", "CMPLNT_TO_DT", "RPT_DT"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_cols].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 579561 entries, 0 to 579560
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   CMPLNT_FR_DT  579550 non-null  datetime64[ns]
 1   CMPLNT_TO_DT  553181 non-null  datetime64[ns]
 2   RPT_DT        579561 non-null  datetime64[ns]
dtypes: datetime64[ns](3)
memory usage: 13.3 MB


In [5]:
#combine date and time columns into datetime columns
df["CMPLNT_FR_DTTM"] = pd.to_datetime(
    df["CMPLNT_FR_DT"].dt.strftime("%Y-%m-%d") + " " + df["CMPLNT_FR_TM"],
    errors="coerce"
)

df["CMPLNT_TO_DTTM"] = pd.to_datetime(
    df["CMPLNT_TO_DT"].dt.strftime("%Y-%m-%d") + " " + df["CMPLNT_TO_TM"],
    errors="coerce"
)
df["IS_2025"] = df["CMPLNT_FR_DT"].dt.year == 2025

print(df[["CMPLNT_FR_DTTM", "CMPLNT_TO_DTTM"]].isna().sum())

CMPLNT_FR_DTTM       11
CMPLNT_TO_DTTM    26545
dtype: int64


In [6]:
# check categorical columns for unexpected values
cols_to_check = [
    "BORO_NM", "CRM_ATPT_CPTD_CD", "LAW_CAT_CD",
    "VIC_SEX", "SUSP_SEX", "VIC_AGE_GROUP", "SUSP_AGE_GROUP",
    "VIC_RACE", "SUSP_RACE"
]

for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(20))


--- BORO_NM ---
BORO_NM
BROOKLYN         162387
MANHATTAN        136768
BRONX            130360
QUEENS           124472
STATEN ISLAND     24215
NaN                1359
Name: count, dtype: int64

--- CRM_ATPT_CPTD_CD ---
CRM_ATPT_CPTD_CD
COMPLETED    572156
ATTEMPTED      7405
Name: count, dtype: int64

--- LAW_CAT_CD ---
LAW_CAT_CD
MISDEMEANOR    299832
FELONY         188142
VIOLATION       91587
Name: count, dtype: int64

--- VIC_SEX ---
VIC_SEX
F    204732
M    183544
E    107542
D     80486
L      3257
Name: count, dtype: int64

--- SUSP_SEX ---
SUSP_SEX
M      311140
U      120751
F       89203
NaN     58467
Name: count, dtype: int64

--- VIC_AGE_GROUP ---
VIC_AGE_GROUP
UNKNOWN    196322
25-44      191838
45-64       97341
18-24       43889
65+         29887
<18         20263
-1              3
-968            2
983             1
-972            1
949             1
951             1
-954            1
937             1
-2              1
942             1
978             1
-967      

In [7]:
#cleaning age group columns to AGE_GROUP_CLEAN
valid_age_groups = {"UNKNOWN", "<18", "18-24", "25-44", "45-64", "65+"}

df["VIC_AGE_GROUP_CLEAN"] = df["VIC_AGE_GROUP"].where(
    df["VIC_AGE_GROUP"].isin(valid_age_groups),
    "UNKNOWN"
)

df["SUSP_AGE_GROUP_CLEAN"] = df["SUSP_AGE_GROUP"].where(
    df["SUSP_AGE_GROUP"].isin(valid_age_groups),
    "UNKNOWN"
)

In [8]:
# check for outliers in numeric columns using IQR method
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

outlier_summary = []

for col in numeric_cols:
    series = df[col].dropna()
    
    if len(series) == 0:
        continue
    
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outlier_mask = (df[col] < lower) | (df[col] > upper)
    outlier_count = outlier_mask.sum()
    
    outlier_summary.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": int(outlier_count),
        "outlier_pct": round(outlier_count / len(df) * 100, 4)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values("outlier_count", ascending=False)
outlier_df

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
2,JURISDICTION_CODE,0.000000,0.000000e+00,0.000000,0.000000,0.000000e+00,77798,13.4236
6,X_COORD_CD,991511.000000,1.017927e+06,26416.000000,951887.000000,1.057551e+06,16335,2.8185
9,Longitude,-73.973824,-7.387845e+01,0.095374,-74.116885,-7.373539e+01,16131,2.7833
1,HOUSING_PSA,477.000000,3.969000e+03,3492.000000,-4761.000000,9.207000e+03,8174,1.4104
0,ADDR_PCT_CD,40.000000,1.010000e+02,61.000000,-51.500000,1.925000e+02,0,0.0000
3,KY_CD,118.000000,3.510000e+02,233.000000,-231.500000,7.005000e+02,0,0.0000
4,PD_CD,259.000000,6.380000e+02,379.000000,-309.500000,1.206500e+03,0,0.0000
5,TRANSIT_DISTRICT,3.000000,3.200000e+01,29.000000,-40.500000,7.550000e+01,0,0.0000
7,Y_COORD_CD,185302.000000,2.358460e+05,50544.000000,109486.000000,3.116620e+05,0,0.0000
8,Latitude,40.675225,4.081399e+01,0.138767,40.467075,4.102214e+01,0,0.0000


In [9]:
#bad latitude and longitude values
geo_bad = df[
    (df["Latitude"] < 40.4) | (df["Latitude"] > 41.1) |
    (df["Longitude"] < -74.5) | (df["Longitude"] > -73.0)
]

print("Geographic outliers:", len(geo_bad))


Geographic outliers: 0


In [10]:
# Save the cleaned dataset to a new CSV file
df.to_csv("NYPD_Complaint_Data_Current_YTD_cleaned.csv", index=False)